# YOLOv6 CRP / PCX Explanation Notebook (Person–Vehicle)

This notebook loads a YOLOv6s6 model and a Person–Car dataset, runs (optional) glocal CRP analysis, and visualizes concept-based explanations and reference images.


## 1) Jupyter autoreload (optional)

Keeps imported modules up to date when you edit code.


In [ ]:
%load_ext autoreload
%autoreload 2

## 2) Silence noisy loggers (run this early)

Prevents Numba / PIL debug spam in the notebook output.


In [ ]:
# --- FIRST CELL (before importing umap/numba!) ---
import os, logging

# Ensure env isn’t forcing DEBUG
os.environ["NUMBA_LOG_LEVEL"] = "WARNING"   # or "ERROR"/"CRITICAL"
os.environ.pop("NUMBA_DEBUG", None)

# Hard‐mute numba loggers and stop propagation to root
for name in ("numba", "numba.core", "numba.core.ssa"):
    lg = logging.getLogger(name)
    lg.setLevel(logging.ERROR)      # try WARNING/ERROR/CRITICAL
    lg.propagate = False
    # remove any existing noisy handlers
    for h in list(lg.handlers):
        lg.removeHandler(h)
    lg.addHandler(logging.NullHandler())

# --- Silence Matplotlib DEBUG logging ---
import os, logging
import matplotlib as mpl

# Just in case someone exported this
os.environ.pop("MPLDEBUG", None)

# Matplotlib's own switch
try:
    mpl.set_loglevel("warning")   # or "error"
except Exception:
    pass

# Force all matplotlib loggers to WARNING and stop propagation to root
for name in ("matplotlib", "matplotlib.font_manager"):
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)   # or logging.ERROR
    lg.propagate = False
    # Remove any existing noisy handlers (e.g., StreamHandler at DEBUG)
    for h in list(lg.handlers):
        lg.removeHandler(h)
    # Add a NullHandler so nothing leaks upward
    lg.addHandler(logging.NullHandler())

# --- Silence PIL/Pillow DEBUG logs ---
import os, logging
os.environ.pop("PILLOW_DEBUG", None)  # just in case

# Force every PIL logger to WARNING (or ERROR) and stop propagation
for name in [n for n in logging.root.manager.loggerDict if n == "PIL" or n.startswith("PIL.")]:
    lg = logging.getLogger(name)
    lg.setLevel(logging.WARNING)      # or logging.ERROR
    lg.propagate = False
    for h in list(lg.handlers):       # remove any noisy handlers
        lg.removeHandler(h)
    lg.addHandler(logging.NullHandler())


## 3) Device selection

Select GPU/CPU device for inference and attribution.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cuda:1"
print("Using device:", device)


## 4) Project path setup

Add the repository root so local modules can be imported.


In [ ]:
import sys

project_root = "/home/said/dev_v1/FHHI-XAI"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

## 5) Imports

Core imports for dataset loading, analysis, and plotting.


In [ ]:
import torch
import torchvision.transforms as transforms
from src.glocal_analysis import run_analysis
from src.plot_crp_explanations import plot_explanations
from src.datasets.person_car_dataset import PersonCarDataset
from LCRP.models import get_model
from src.letterbox_utils import letterbox_transform
import LCRP.models.yolov6 as yolov6

## 6) Load dataset and model


In [ ]:
dtype = torch.float32
# root_dir = "../YOLOV6/data/synthetic/"
root_dir = "../YOLOV6/data/original/"

from functools import partial
transform = partial(letterbox_transform, target_size=640, stride=64, half=False, auto=False)
dataset = PersonCarDataset(root_dir=root_dir, split="train", transform=transform)

model_name = "yolov6s6"
# ckpt_path = "../YOLOV6/weights/trial/best_ckpt_synthetic.pt"
ckpt_path = "../YOLOV6/weights/trial/best_ckpt_original.pt"

# Loading unet with path to checkpoint
model = get_model(model_name=model_name, classes=2, ckpt_path=ckpt_path, device=device, dtype=dtype)
model = model.to(device)

### Sanity check


In [ ]:
print(len(dataset))

## 7) Run analysis and visualize explanations


In [ ]:
# This folder contains results of the glocal analysis.
# output_dir = "../output_synthetic/crp/yolo_person_car"
output_dir = "../output_original/crp/yolo_person_car"
# output_dir = "../output/crp/yolo_person_car"


### Run glocal analysis (slow; optional)


In [ ]:
# # Only run this if needed, takes a long time (79 on cpu on my laptop).
# run_analysis(model_name, model, dataset, output_dir=output_dir, device=device)

### Debug: verify saved maximization indices (optional)


In [ ]:
# # Check the saved_checkpoints or if there were errors
# # Look at what indices are actually in the maximization files
# 
# import numpy as np
# 
# crp_path = "../output_original/crp/yolo_person_car"
# layer_name = "module.backbone.stem.rbr_dense.conv"  # use one from your list
# 
# # Load and check which dataset indices were saved
# d_c_sorted = np.load(f"{crp_path}/RelMax_sum_normed/{layer_name}_data.npy")
# print(f"Shape: {d_c_sorted.shape}")
# print(f"Dataset indices in file (first column): {np.unique(d_c_sorted[:, 0])}")
# print(f"Min index: {d_c_sorted.min()}, Max index: {d_c_sorted.max()}")

## 8) Generate / save reference images for concept prototypes (optional)


In [ ]:
# import os
# from crp.helper import load_maximization, get_layer_names
# from LCRP.utils.render import vis_opaque_img_border
# from crp.concepts import ChannelConcept
# from src.pcx_helper import get_ref_images
# import torch
# from LCRP.utils.crp_configs import ATTRIBUTORS, CANONIZERS, VISUALIZATIONS, COMPOSITES
# 
# # Setup
# model_name = "yolov6s6"
# output_dir_crp = "../output_original/crp/yolo_person_car"
# ref_imgs_path_12 = "../output_original/ref_imgs/ref_imgs_12/"
# ref_imgs_path_6 = "../output_original/ref_imgs/ref_imgs_6/"
# 
# # ============================================================
# # SPECIFY YOUR LAYERS HERE
# # ============================================================
# layers_to_process = [
#     "module.backbone.stem.rbr_dense.conv",
#     "module.backbone.ERBlock_2.0.rbr_dense.conv",
#     "module.backbone.ERBlock_3.0.rbr_dense.conv",
# ]
# 
# # Classes to process
# class_ids = [1]  # 0=person, 1=car
# 
# # Number of reference images to save
# n_refs = [12, 6]  # Will save both 12 and 6 ref images
# 
# # ============================================================
# # Setup CRP components
# # ============================================================
# layer_names_all = get_layer_names(model, types=[torch.nn.Conv2d])
# attribution = ATTRIBUTORS[model_name](model)
# composite = COMPOSITES[model_name](canonizers=[CANONIZERS[model_name]()])
# 
# fv = VISUALIZATIONS[model_name](
#     attribution, dataset, layer_names_all,
#     preprocess_fn=lambda x: x,
#     path=output_dir_crp,
#     max_target="max"
# )
# 
# # ============================================================
# # Process each layer
# # ============================================================
# crp_relmax_path = os.path.join(output_dir_crp, "RelMax_sum_normed")
# 
# for layer_name in layers_to_process:
#     print(f"\n{'='*60}")
#     print(f"Processing layer: {layer_name}")
#     print(f"{'='*60}")
# 
#     try:
#         # Load maximization data to get concept count
#         d_c_sorted, _, _ = load_maximization(crp_relmax_path, layer_name)
#         n_concepts = d_c_sorted.shape[1]
#         print(f"  Layer has {n_concepts} concepts, {d_c_sorted.shape[0]} samples")
# 
#         # All concept indices for this layer
#         all_concepts = list(range(n_concepts))
# 
#         for class_id in class_ids:
#             for n_ref in n_refs:
#                 ref_path = ref_imgs_path_12 if n_ref == 12 else ref_imgs_path_6
# 
#                 print(f"\n  Class {class_id}, n_ref={n_ref}...")
# 
#                 try:
#                     ref_imgs = get_ref_images(
#                         fv, all_concepts, layer_name,
#                         composite=composite,
#                         class_id=class_id,
#                         n_ref=n_ref,
#                         ref_imgs_save_path=ref_path
#                     )
#                     print(f"    ✓ Saved {len(ref_imgs)} concepts")
#                 except Exception as e:
#                     print(f"    ❌ Error: {e}")
# 
#     except Exception as e:
#         print(f"  ❌ Error loading layer: {e}")
# 
# print("\n" + "="*60)
# print("DONE!")
# print("="*60)

## 9) Plot explanations for a specific sample

Edit the parameters below and run the `plot_explanations(...)` call.


In [ ]:
# # Setting up main parameters
# class_id = 1 #1/2
# sample_id = 271 #data number in the dataset
# n_concepts =  3 #jj
# n_refimgs = 12 # constant
# layer = "module.backbone.ERBlock_3.0.rbr_dense.conv" # change the layer
# mode = "relevance"
# prediction_num = 0 
# 
# # if failing, try to restart the notebook and do not run analysis again, go directly to plotting
# plot_explanations(model_name, model, dataset, sample_id, class_id, layer, prediction_num, mode, n_concepts, n_refimgs, output_dir=output_dir)

## 10) Cleanup (use with caution)

terminate Jupyter processes.


In [ ]:
!pkill -u said -f jupyter
